In [1]:
import pandas as pd
import duckdb

# 设备每日运行日志表
log_data = [
    ["A", "2026-07-01", "NORMAL", 2],
    ["A", "2026-07-02", "ERROR", 5],
    ["A", "2026-07-03", "ERROR", 6],

    ["B", "2026-07-01", "NORMAL", 1],
    ["B", "2026-07-02", "NORMAL", 2],
    ["B", "2026-07-03", "ERROR", 4],

    ["C", "2026-07-01", "ERROR", 7],
    ["C", "2026-07-02", "NORMAL", 3],

    # E 在日志表中存在，但设备信息表中没有
    ["E", "2026-07-01", "ERROR", 9],
    ["E", "2026-07-02", "ERROR", 8],
]

df_log = pd.DataFrame(
    log_data,
    columns=["device_id", "stat_date", "status", "alarm_count"]
)

df_log["stat_date"] = pd.to_datetime(df_log["stat_date"])


# 设备基础信息表
device_data = [
    ["A", "R34", "VIS", "FS11"],
    ["B", "R34", "RVR", "LT31"],
    ["C", "R35", "VIS", "FS11"],

    # D 在设备信息表中存在，但日志表中没有
    ["D", "R35", "RVR", "LT31"],
]

df_device = pd.DataFrame(
    device_data,
    columns=["device_id", "site", "device_type", "model"]
)

print("df_log:")
print(df_log)

print("\ndf_device:")
print(df_device)

df_log:
  device_id  stat_date  status  alarm_count
0         A 2026-07-01  NORMAL            2
1         A 2026-07-02   ERROR            5
2         A 2026-07-03   ERROR            6
3         B 2026-07-01  NORMAL            1
4         B 2026-07-02  NORMAL            2
5         B 2026-07-03   ERROR            4
6         C 2026-07-01   ERROR            7
7         C 2026-07-02  NORMAL            3
8         E 2026-07-01   ERROR            9
9         E 2026-07-02   ERROR            8

df_device:
  device_id site device_type model
0         A  R34         VIS  FS11
1         B  R34         RVR  LT31
2         C  R35         VIS  FS11
3         D  R35         RVR  LT31


## Task 1：按站点统计总报警次数

* **统计每个站点的总报警次数。**

## 输出字段：

- `site`
- `total_alarm_count`

## 要求：

- 只统计能够匹配到设备信息的日志记录。
- 无法匹配到 site 的日志记录不要计入统计。

* **也就是说，设备 E 的日志不应该参与站点统计，因为它在设备信息表中没有站点信息。**

In [ ]:
# ==================
# Task1(SQL轨道)
# ==================

query = """

SELECT
    dv.site,
    SUM(lo.alarm_count)::INTEGER AS total_alarm_count
FROM df_log AS lo
INNER JOIN df_device AS dv
    ON lo.device_id = dv.device_id
GROUP BY dv.site
ORDER BY dv.site;
"""
df_sql = duckdb.execute(query).fetchdf()
df_sql

,site,alarm_count
0,R34,20
1,R35,10


In [ ]:
# ==================
# Task1(PANDAS轨道)
# ==================

df_pd = (
    df_log
    .merge(
        df_device,
        how='inner',
        on='device_id'
    )
    .groupby('site', as_index=False)
    .agg(
        total_alarm_count=('alarm_count', 'sum')
    )
    .sort_values(by='site')
    .reset_index(drop=True)
)

df_pd

,site,alarm_count
0,R34,20
1,R35,10


## Task2

**要求：**

- 只统计能够匹配到设备信息的日志记录。

- 按 `site` 和 `device_type` 两个字段分组。

* **输出字段：**

- `site`
- `device_type`


- `total_alarm_count：`
该站点、该设备类型的总报警次数。

- `error_days：`
该站点、该设备类型下 status = 'ERROR' 的记录数。

- `avg_alarm_count：`
该站点、该设备类型的平均每日报警次数，保留 2 位小数。

- `rank_num`

In [ ]:
# ==================
# Task2(SQL轨道)
# ==================

query = """

WITH merge_table AS (

    SELECT
        lo.device_id,
        lo.stat_date,
        lo.status,
        lo.alarm_count,
        dv.site,
        dv.device_type
    FROM df_log AS lo
    INNER JOIN df_device AS dv
        ON lo.device_id = dv.device_id
)

SELECT 
    site,
    device_type,
    SUM(alarm_count)::INTEGER AS total_alarm_count,
    COUNT(*) FILTER (WHERE status = 'ERROR') AS error_days,
    ROUND(AVG(alarm_count), 2) AS avg_alarm_count
FROM merge_table
GROUP BY site, device_type
ORDER BY site, device_type;
"""
df_sql = duckdb.execute(query).fetchdf()
df_sql

,site,device_type,total_alarm_count,error_days,avg_alarm_count
0,R34,RVR,7,1,2.33
1,R34,VIS,13,2,4.33
2,R35,VIS,10,1,5.00


In [40]:
# ==================
# Task2(PANDAS轨道)
# ==================

df_task2 = (
    df_log
    .merge(
        df_device,
        how='inner',
        on='device_id'
    )
    .assign(
        is_error=lambda x: x['status'] == 'ERROR'
    )
    .groupby(['site', 'device_type'], as_index=False)
    .agg(
        total_alarm_count=('alarm_count', 'sum'),
        error_days=('is_error', 'sum'),
        avg_alarm_count=('alarm_count', 'mean')
    )
    .assign(
        avg_alarm_count=lambda x: x['avg_alarm_count'].round(2)
    )
    .sort_values(by=['site', 'device_type'])
    .reset_index(drop=True)
)

df_task2

,site,device_type,total_alarm_count,error_days,avg_alarm_count
0,R34,RVR,7,1,2.33
1,R34,VIS,13,2,4.33
2,R35,VIS,10,1,5.00
